# AFL Analytics — Exploratory Data Analysis
Connects to the SQLite star schema and profiles the data.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
DB_PATH = '../data/database/afl_analytics.db'
conn = sqlite3.connect(DB_PATH)

In [ ]:
# Check row counts across tables
tables = ['dim_players','dim_teams','dim_venues','dim_seasons','dim_date',
          'fact_player_match_stats','fact_match_results']
for t in tables:
    try:
        n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {t}', conn).iloc[0,0]
        print(f'{t:<30} {n:>8,} rows')
    except Exception as e:
        print(f'{t:<30} ERROR: {e}')

In [ ]:
# Games per season
df = pd.read_sql("""
    SELECT s.year, COUNT(*) AS matches
    FROM fact_match_results f
    JOIN dim_seasons s ON f.season_key = s.season_key
    GROUP BY s.year ORDER BY s.year
""", conn)
df.plot(x='year', y='matches', kind='bar', figsize=(12,4), title='Matches per Season')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of disposals
df = pd.read_sql('SELECT disposals FROM fact_player_match_stats WHERE disposals IS NOT NULL', conn)
df['disposals'].plot(kind='hist', bins=50, figsize=(10,4), title='Distribution of Disposals per Game')
plt.xlabel('Disposals')
plt.tight_layout()
plt.show()
df['disposals'].describe()

In [ ]:
# Average team score by season
df = pd.read_sql("""
    SELECT s.year,
           ROUND(AVG(f.home_total_score),1) AS avg_home,
           ROUND(AVG(f.away_total_score),1) AS avg_away
    FROM fact_match_results f
    JOIN dim_seasons s ON f.season_key = s.season_key
    GROUP BY s.year ORDER BY s.year
""", conn)
df.set_index('year').plot(figsize=(12,4), title='Average Score by Season (Home vs Away)')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 players by average disposals (min 50 games)
df = pd.read_sql("""
    SELECT p.player_name, t.team_name,
           COUNT(*) AS games,
           ROUND(AVG(f.disposals),1) AS avg_disposals
    FROM fact_player_match_stats f
    JOIN dim_players p ON f.player_key = p.player_key
    JOIN dim_teams   t ON f.team_key   = t.team_key
    WHERE f.disposals IS NOT NULL
    GROUP BY f.player_key, f.team_key
    HAVING games >= 50
    ORDER BY avg_disposals DESC
    LIMIT 10
""", conn)
print(df.to_string(index=False))

In [ ]:
conn.close()